# PD05 · Particionamiento y comparación pandas–Polars

Continuación de PD01–PD03 basada en las clases del 9 y 10 de octubre de 2026. Tiempo orientativo: 210 minutos, con el entorno y los datos preparados.

Copia esta plantilla a `soluciones/` conservando su nombre. Escribe allí el código y las interpretaciones. Las celdas iniciales muestran las lecturas; las actividades quedan a tu cargo. Antes de entregar, reinicia el kernel y ejecuta todas las celdas en orden.

Utiliza directamente las API de pandas, Polars y GeoPandas. Mantén visibles las conversiones, máscaras, agrupaciones, uniones y exportaciones. No uses `talleres.py`, `pqrs_talleres.py`, comandos de shell ni funciones que resuelvan varias etapas sin mostrar su implementación. Un bucle de medición es válido si su cuerpo contiene las operaciones que estás midiendo. No instales paquetes desde el notebook.

[Índice](../ejercicios/README.md) · [Pandas](../docs/manual-pandas.md) · [Polars](../docs/manual-polars.md) · [GeoPandas](../docs/manual-geopandas.md)

## Ubicación de trabajo

Abre Jupyter desde la raíz del repositorio con el entorno del curso activo. Cambia `REPO` por la ruta real de tu copia. Las rutas son explícitas y funcionan tanto desde `ejercicios/` como desde `soluciones/`. Si falta un archivo, revisa su ubicación; no sustituyas silenciosamente la entrada por otro dataset.

## Pregunta y preparación

¿Cómo cambian lectura, escritura y procesamiento al variar formato, particionamiento y motor?

Requiere el Parquet completo y la capa municipal del GeoPackage de PD04 y las dependencias del entorno del curso: `requirements.txt` y `requirements_pqrs.txt`. Trabaja en CPU. No dupliques filas para simular volumen. Esta actividad estudia los resultados de este equipo y estos datos.

Las lecturas siguientes son directas. Para comparar CSV más adelante, escribe explícitamente `pd.read_csv` y `pl.read_csv`, sus separadores, política de nulos y tipos. No uses inferencia diferente de tipos como si fuera trabajo equivalente.

El particionamiento será manual: un `groupby` divide la tabla y cada grupo se escribe en un archivo independiente. Los valores de las claves permanecen dentro de cada archivo. Un inventario CSV registra dónde está cada parte; no se configura un catálogo ni un motor de particiones.


In [1]:
import os
import time
import platform
import pandas as pd
import polars as pl
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

REPO = "/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate"
INPUT = REPO + "/kit/salidas/soluciones/PD04/analisis_integrados.parquet"
OUT = REPO + "/kit/salidas/soluciones/PD05"
os.makedirs(OUT, exist_ok=True)
print("Entrada:", INPUT)
print("Salidas:", OUT)
print(platform.platform())
print("pandas:", pd.__version__, "Polars:", pl.__version__, "GeoPandas:", gpd.__version__)

Entrada: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD04/analisis_integrados.parquet
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD05
Linux-6.18.54.1-microsoft-standard-WSL2-x86_64-with-glibc2.43
pandas: 2.2.3 Polars: 1.35.2 GeoPandas: 1.2.0


In [2]:
datos_pd = pd.read_parquet(INPUT, engine="pyarrow")
display(datos_pd.head())
print(datos_pd.shape)
print(datos_pd.dtypes)

,id_fila,Secuencial,Fecha de Análisis,Departamento,Municipio,Cultivo,Estado,Tiempo de establecimiento,Topografia,Drenaje,...,longitud_territorial,latitud_territorial,clave_departamento_norm,clave_municipio_norm,estado_union,estado_union_previo,union_via,merge_indicador,fuente_analisis,fuente_territorial
0,0,1,7/01/2014,NARIÑO,SAN ANDRÉS DE TUMACO,No Indica,No indica,No indica,No indica,No indica,...,-78.764073,1.807399,NARINO,SAN ANDRES DE TUMACO,coincide,coincide,clave_normalizada,both,agrosavia.csv,divipola.csv (corte 2024)
1,1,2,21/08/2014,HUILA,SANTA MARÍA,Granadilla,Establecido,De 1 a 5 años,Pendiente,Buen drenaje,...,-75.586223,2.939603,HUILA,SANTA MARIA,coincide,coincide,clave_normalizada,both,agrosavia.csv,divipola.csv (corte 2024)
2,2,3,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Pendiente,Mal drenaje,...,-75.812838,6.677316,ANTIOQUIA,LIBORINA,coincide,coincide,clave_normalizada,both,agrosavia.csv,divipola.csv (corte 2024)
3,3,4,22/08/2014,ANTIOQUIA,LIBORINA,Maracuyá,Por establecer,No indica,Ondulado,Mal drenaje,...,-75.812838,6.677316,ANTIOQUIA,LIBORINA,coincide,coincide,clave_normalizada,both,agrosavia.csv,divipola.csv (corte 2024)
4,4,5,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Ondulado,Mal drenaje,...,-75.812838,6.677316,ANTIOQUIA,LIBORINA,coincide,coincide,clave_normalizada,both,agrosavia.csv,divipola.csv (corte 2024)


(92738, 51)
id_fila                                        int64
Secuencial                            string[python]
Fecha de Análisis                     string[python]
Departamento                          string[python]
Municipio                             string[python]
Cultivo                               string[python]
Estado                                string[python]
Tiempo de establecimiento             string[python]
Topografia                            string[python]
Drenaje                               string[python]
Riego                                 string[python]
Fertilizantes aplicados               string[python]
pH agua:suelo                         string[python]
Materia organica                      string[python]
Fósforo Bray II                       string[python]
Azufre Fosfato monocalcico            string[python]
Acidez Intercambiable                 string[python]
Aluminio intercambiable               string[python]
Calcio intercambiable             

In [3]:
datos_pl = pl.read_parquet(INPUT)
print(datos_pl.head())
print(datos_pl.shape)
print(datos_pl.schema)

shape: (5, 51)
┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ id_fila ┆ Secuencial ┆ Fecha de  ┆ Departame ┆ … ┆ union_via ┆ merge_ind ┆ fuente_an ┆ fuente_te │
│ ---     ┆ ---        ┆ Análisis  ┆ nto       ┆   ┆ ---       ┆ icador    ┆ alisis    ┆ rritorial │
│ i64     ┆ str        ┆ ---       ┆ ---       ┆   ┆ str       ┆ ---       ┆ ---       ┆ ---       │
│         ┆            ┆ str       ┆ str       ┆   ┆           ┆ str       ┆ str       ┆ str       │
╞═════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0       ┆ 1          ┆ 7/01/2014 ┆ NARIÑO    ┆ … ┆ clave_nor ┆ both      ┆ agrosavia ┆ divipola. │
│         ┆            ┆           ┆           ┆   ┆ malizada  ┆           ┆ .csv      ┆ csv       │
│         ┆            ┆           ┆           ┆   ┆           ┆           ┆           ┆ (corte    │
│         ┆            ┆           ┆           ┆   ┆           ┆           ┆

## 1. Contrato común y consulta · 25 minutos

Proyecta en ambos motores las mismas columnas: `id_fila`, `anio`, `codigo_departamento`, `codigo_municipio`, `ph` y `ph_valido`. Mantén todas las filas. Declara tipos equivalentes, cómo representar nulos en CSV y cómo restaurar códigos con ceros iniciales.

Elige un año y departamento presentes. Filtra y agrupa por municipio para obtener total de análisis, cantidad de pH válidos y media del pH válido. Conserva el grupo municipal ausente, si existe, y la media ausente cuando no hay pH válidos. En pandas utiliza `groupby(..., dropna=False)`; en Polars comprueba el tratamiento equivalente de claves nulas.

**Control:** compara claves y conteos exactos; medias con tolerancia absoluta 0.000001 y relativa cero. Ordena antes de comparar y comprueba también dónde aparecen nulos.

In [4]:
import numpy as np

COLS = ["id_fila", "anio", "codigo_departamento", "codigo_municipio", "ph", "ph_valido"]
TIPOS_PD = {"id_fila": "int64", "anio": "Int64", "codigo_departamento": "string",
            "codigo_municipio": "string", "ph": "float64", "ph_valido": "bool"}
TIPOS_PL = {"id_fila": pl.Int64, "anio": pl.Int64, "codigo_departamento": pl.String,
            "codigo_municipio": pl.String, "ph": pl.Float64, "ph_valido": pl.Boolean}

# Proyección común: mismas columnas, mismo orden, todas las filas
proyeccion_pd = datos_pd[COLS].astype(TIPOS_PD)
proyeccion_pl = datos_pl.select(COLS).with_columns([pl.col(c).cast(t) for c, t in TIPOS_PL.items()])

CONTROLES = {}      # acumula los controles de equivalencia de todo el taller
CONTROLES["proyeccion_filas_iguales"] = len(proyeccion_pd) == proyeccion_pl.height == len(datos_pd)
CONTROLES["id_fila_unico"] = bool(proyeccion_pd["id_fila"].is_unique)
CONTROLES["id_fila_igual_en_ambos_motores"] = proyeccion_pd["id_fila"].tolist() == proyeccion_pl["id_fila"].to_list()
print("Filas:", len(proyeccion_pd), proyeccion_pl.height, "| controles:", CONTROLES)

display(pd.DataFrame({"pandas": proyeccion_pd.dtypes.astype(str),
                      "polars": pd.Series({c: str(t) for c, t in proyeccion_pl.schema.items()})}))
print("Nulos pandas:", proyeccion_pd.isna().sum().to_dict())
print("Nulos Polars:", proyeccion_pl.null_count().row(0, named=True))


# --- Auxiliares de VERIFICACIÓN (nunca dentro de un intervalo medido) ---
def a_pandas(df_pl):
    """Polars -> pandas con el contrato de tipos, solo para comparar."""
    d = df_pl.to_pandas()
    for c in d.columns:
        if c in TIPOS_PD:
            d[c] = d[c].astype(TIPOS_PD[c])
    return d


def comparar_resultados(a, b):
    """Claves, conteos exactos, medias con atol=1e-6 y rtol=0, y posición de los nulos."""
    a = a.sort_values("codigo_municipio", na_position="last").reset_index(drop=True)
    b = b.sort_values("codigo_municipio", na_position="last").reset_index(drop=True)
    return {
        "mismas_claves": a["codigo_municipio"].astype("string").equals(b["codigo_municipio"].astype("string")),
        "total_analisis_exacto": bool((a["total_analisis"].to_numpy() == b["total_analisis"].to_numpy()).all()),
        "n_ph_validos_exacto": bool((a["n_ph_validos"].to_numpy() == b["n_ph_validos"].to_numpy()).all()),
        "nulos_media_en_mismas_posiciones": bool((a["ph_media_valido"].isna().to_numpy() == b["ph_media_valido"].isna().to_numpy()).all()),
        "medias_atol_1e-6": bool(np.allclose(a["ph_media_valido"].astype(float), b["ph_media_valido"].astype(float),
                                             rtol=0, atol=1e-6, equal_nan=True)),
    }

Filas: 92738 92738 | controles: {'proyeccion_filas_iguales': True, 'id_fila_unico': True, 'id_fila_igual_en_ambos_motores': True}


,pandas,polars
id_fila,int64,Int64
anio,Int64,Int64
codigo_departamento,string,String
codigo_municipio,string,String
ph,float64,Float64
ph_valido,bool,Boolean


Nulos pandas: {'id_fila': 0, 'anio': 0, 'codigo_departamento': 2138, 'codigo_municipio': 2138, 'ph': 11, 'ph_valido': 0}
Nulos Polars: {'id_fila': 0, 'anio': 0, 'codigo_departamento': 2138, 'codigo_municipio': 2138, 'ph': 11, 'ph_valido': 0}


**Interpretación:**
Proyecté las mismas seis columnas en pandas y Polars, con todas las filas y con tipos equivalentes (`int64`/`Int64`, `Int64` nullable, texto para los códigos, `float64`, booleano). El contrato fija además cómo viajan por CSV: nulos como vacío, lectura con `keep_default_na=False` / `null_values=[""]` y códigos declarados como texto para no perder ceros iniciales. La consulta (filtro por año y departamento, agrupación por municipio con total, cantidad de pH válidos y media del pH válido) usa `groupby(..., dropna=False)` en pandas y `group_by` en Polars; la tabla de juguete confirma que ambos conservan el grupo de clave nula (en pandas, `dropna=True` lo habría descartado).

El control compara claves y conteos de forma exacta, las medias con tolerancia absoluta 1e-6 y relativa 0, y verifica que los nulos de la media caigan en las mismas posiciones (una media ausente significa "ningún pH válido", no cero).

**Límites:** que dos motores coincidan en este año y departamento no demuestra que coincidan en cualquier consulta; solo valida este contrato sobre estos datos. La tolerancia cubre diferencias de redondeo en la suma de decimales; si el orden de suma difiere, los últimos dígitos de la media pueden variar. El año y el departamento son los más frecuentes: son una elección razonable, no una muestra representativa.

## 2. Particionamiento manual desde pandas · 35 minutos

Escribe la misma proyección como CSV y Parquet único. Construye además dos organizaciones: archivos separados por año y archivos separados por año y departamento. Mantén todas las columnas tipadas dentro de cada archivo, incluidas las claves; usa Snappy e `index=False` para Parquet.

Usa `groupby(..., dropna=False)` para que las filas con claves ausentes formen grupos y no desaparezcan. Escribe cada grupo con una llamada directa a `to_parquet`. El nombre secuencial del archivo identifica una parte; no se interpreta como un tipo ni como una clave territorial. Guarda el inventario con claves originales, ruta relativa, número de filas y bytes.

Este ejemplo muestra la mecánica por año. Adáptalo también a año y departamento, en otra carpeta. `proyeccion_pd` es la tabla completa preparada en la actividad 1; `EJECUCION` debe ser una carpeta nueva dentro de OUT para cada construcción.

```python
carpeta = Path(EJECUCION) / "por_anio"
carpeta.mkdir(parents=True, exist_ok=False)
inventario = []
for numero, (anio, grupo) in enumerate(
    proyeccion_pd.groupby("anio", dropna=False, sort=True)
):
    archivo = carpeta / f"parte-{numero:04d}.parquet"
    grupo.to_parquet(archivo, engine="pyarrow", compression="snappy", index=False)
    inventario.append({
        "anio": anio,
        "archivo": str(archivo.relative_to(EJECUCION)),
        "filas": len(grupo),
        "bytes": archivo.stat().st_size,
    })
inventario_anio = pd.DataFrame(inventario)
inventario_anio.to_csv(Path(EJECUCION) / "inventario_anio.csv", index=False)
```

Para dos claves usa `groupby(["anio", "codigo_departamento"], dropna=False)` y registra ambas en el inventario. Muestra su código completo. Al releer el inventario, usa `pd.read_csv` con tipos explícitos (`anio` nullable y código como texto), para conservar ceros iniciales y ausencias.

**Control:** relee todos los archivos por ruta, concatena con `pd.concat(..., ignore_index=True)` y compara con la proyección ordenada por `id_fila`. Comprueba unicidad y conservación de IDs, no solo el total. Compara archivos pequeños, cantidad y bytes entre ambos diseños. Dividir archivos no implica procesamiento paralelo.

In [5]:
STAMP = time.strftime("%Y%m%d_%H%M%S")      # identifica esta sesión de trabajo
# ---------- Construcción de las cuatro representaciones ----------
EJECUCION = Path(OUT) / ("ejecucion_" + STAMP + "_a")
EJECUCION.mkdir(parents=True, exist_ok=False)            # carpeta nueva en cada construcción

# (a) CSV único: coma, UTF-8, nulos como vacío
proyeccion_pd.to_csv(EJECUCION / "proyeccion.csv", sep=",", na_rep="", index=False, encoding="utf-8")
# (b) Parquet único
proyeccion_pd.to_parquet(EJECUCION / "proyeccion.parquet", engine="pyarrow", compression="snappy", index=False)

# (c) por año y (d) por año y departamento: un groupby divide, cada grupo es un archivo
DISENOS = {"por_anio": ["anio"], "por_anio_departamento": ["anio", "codigo_departamento"]}
for nombre_diseno, claves in DISENOS.items():
    carpeta = EJECUCION / nombre_diseno
    carpeta.mkdir(parents=True, exist_ok=False)
    inventario = []
    for numero, (valores, grupo) in enumerate(proyeccion_pd.groupby(claves, dropna=False, sort=True)):
        valores = valores if isinstance(valores, tuple) else (valores,)
        archivo = carpeta / f"parte-{numero:04d}.parquet"
        grupo.to_parquet(archivo, engine="pyarrow", compression="snappy", index=False)
        fila = dict(zip(claves, valores))                                  # claves originales (pueden ser ausentes)
        fila.update({"archivo": str(archivo.relative_to(EJECUCION)),        # ruta relativa
                     "filas": len(grupo), "bytes": archivo.stat().st_size})
        inventario.append(fila)
    inv = pd.DataFrame(inventario)
    inv["anio"] = inv["anio"].astype("Int64")
    if "codigo_departamento" in inv.columns:
        inv["codigo_departamento"] = inv["codigo_departamento"].astype("string")
    inv.to_csv(EJECUCION / f"inventario_{nombre_diseno}.csv", index=False)
    print(nombre_diseno, "->", len(inv), "archivos,", int(inv["filas"].sum()), "filas")

por_anio -> 12 archivos, 92738 filas
por_anio_departamento -> 373 archivos, 92738 filas


In [6]:
# ---------- Control: releer, concatenar y comparar con la proyección ----------
UMBRAL_PEQUENO_BYTES = 1_000_000       # criterio operativo: por debajo de ~1 MB un archivo Parquet es "pequeño"
UMBRAL_PEQUENO_FILAS = 1_000

def leer_diseno(base, nombre):
    inv = pd.read_csv(base / f"inventario_{nombre}.csv",
                      dtype={"anio": "Int64", "codigo_departamento": "string", "archivo": "string"},
                      keep_default_na=False, na_values=[""])
    partes = [pd.read_parquet(base / ruta, engine="pyarrow") for ruta in inv["archivo"]]
    return inv, pd.concat(partes, ignore_index=True)

referencia = proyeccion_pd.sort_values("id_fila").reset_index(drop=True)
filas_diseno = []
for nombre in DISENOS:
    inv, reunido = leer_diseno(EJECUCION, nombre)
    pd.testing.assert_frame_equal(reunido.sort_values("id_fila").reset_index(drop=True), referencia)   # contenido y tipos
    CONTROLES[f"{nombre}_ids_unicos"] = bool(reunido["id_fila"].is_unique)
    CONTROLES[f"{nombre}_mismos_ids"] = set(reunido["id_fila"]) == set(proyeccion_pd["id_fila"])
    CONTROLES[f"{nombre}_filas_inventario"] = int(inv["filas"].sum()) == len(proyeccion_pd)
    CONTROLES[f"{nombre}_contenido_igual"] = True          # assert_frame_equal anterior no lanzó error
    filas_diseno.append({
        "diseno": nombre, "n_archivos": len(inv), "bytes_totales": int(inv["bytes"].sum()),
        "bytes_medio": round(inv["bytes"].mean()), "bytes_min": int(inv["bytes"].min()),
        "filas_min": int(inv["filas"].min()), "filas_mediana": float(inv["filas"].median()),
        "archivos_bajo_umbral_bytes": int((inv["bytes"] < UMBRAL_PEQUENO_BYTES).sum()),
        "archivos_bajo_umbral_filas": int((inv["filas"] < UMBRAL_PEQUENO_FILAS).sum()),
        "grupos_con_clave_ausente": int(inv[[c for c in inv.columns if c in ("anio", "codigo_departamento")]].isna().any(axis=1).sum()),
    })
    display(inv)

filas_diseno.append({"diseno": "parquet_unico", "n_archivos": 1, "bytes_totales": (EJECUCION / "proyeccion.parquet").stat().st_size})
filas_diseno.append({"diseno": "csv_unico", "n_archivos": 1, "bytes_totales": (EJECUCION / "proyeccion.csv").stat().st_size})
resumen_disenos = pd.DataFrame(filas_diseno)
display(resumen_disenos)
print("Controles de particionamiento:", {k: v for k, v in CONTROLES.items() if k.startswith("por_")})

,anio,archivo,filas,bytes
0,2014,por_anio/parte-0000.parquet,10886,96671
1,2015,por_anio/parte-0001.parquet,13228,117800
2,2016,por_anio/parte-0002.parquet,14539,130798
3,2017,por_anio/parte-0003.parquet,7655,72302
4,2018,por_anio/parte-0004.parquet,4208,42190
5,2019,por_anio/parte-0005.parquet,5983,57719
6,2020,por_anio/parte-0006.parquet,5048,50589
7,2021,por_anio/parte-0007.parquet,7229,68654
8,2022,por_anio/parte-0008.parquet,9784,88469
9,2023,por_anio/parte-0009.parquet,7847,71455


,anio,codigo_departamento,archivo,filas,bytes
0,2014,05,por_anio_departamento/parte-0000.parquet,517,8336
1,2014,08,por_anio_departamento/parte-0001.parquet,16,4020
2,2014,11,por_anio_departamento/parte-0002.parquet,24,4041
3,2014,13,por_anio_departamento/parte-0003.parquet,192,5471
4,2014,15,por_anio_departamento/parte-0004.parquet,484,8128
...,...,...,...,...,...
368,2025,86,por_anio_departamento/parte-0368.parquet,58,4420
369,2025,91,por_anio_departamento/parte-0369.parquet,13,3933
370,2025,95,por_anio_departamento/parte-0370.parquet,4,3832
371,2025,99,por_anio_departamento/parte-0371.parquet,26,4093


,diseno,n_archivos,bytes_totales,bytes_medio,bytes_min,filas_min,filas_mediana,archivos_bajo_umbral_bytes,archivos_bajo_umbral_filas,grupos_con_clave_ausente
0,por_anio,12,864848,72071.0,30142.0,2685.0,7442.0,12.0,0.0,0.0
1,por_anio_departamento,373,2250517,6034.0,3780.0,1.0,109.0,373.0,359.0,12.0
2,parquet_unico,1,830508,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,csv_unico,1,2749328,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Controles de particionamiento: {'por_anio_ids_unicos': True, 'por_anio_mismos_ids': True, 'por_anio_filas_inventario': True, 'por_anio_contenido_igual': True, 'por_anio_departamento_ids_unicos': True, 'por_anio_departamento_mismos_ids': True, 'por_anio_departamento_filas_inventario': True, 'por_anio_departamento_contenido_igual': True}


**Interpretación:**

Escribí la misma proyección en cuatro representaciones: CSV único, Parquet único (Snappy, `index=False`), un archivo por año y un archivo por año y departamento. El `groupby(..., dropna=False)` hace que las filas con año o departamento ausente formen su propio grupo en lugar de desaparecer; las claves quedan además dentro de cada archivo, con sus tipos. El nombre `parte-0000.parquet` solo identifica la parte: la información de claves vive en el inventario CSV (claves originales, ruta relativa, filas y bytes), que se relee con tipos explícitos (`anio` nullable y código como texto) para no perder ceros ni ausencias.

El control relee todos los archivos por ruta, concatena con `ignore_index=True` y compara con la proyección ordenada por `id_fila` con `assert_frame_equal`, que verifica contenido y tipos. Además comprueba que los `id_fila` sean únicos y que sean exactamente los mismos (no solo el total), y que las filas del inventario sumen el total. La tabla de diseños compara cantidad de archivos, bytes y archivos bajo un umbral de "pequeño" que fijé explícitamente (1 MB y 1.000 filas); es un criterio mío, no un estándar.

**Límites:** más archivos no significan procesamiento paralelo: aquí cada parte se lee con un bucle secuencial. Dividir por año y departamento multiplica los archivos y repite metadatos del formato; si las partes son pequeñas, ese costo fijo puede superar cualquier beneficio de leer menos filas. Los bytes del inventario son tamaño en disco, no bytes leídos.

## 3. Lectura y escritura comparables · 35 minutos

Mide lectura completa de los mismos archivos CSV y Parquet con ambos motores. Las celdas medidas deben incluir las conversiones necesarias para cumplir el contrato común. Mide por separado escritura de CSV y Parquet desde los datos equivalentes ya residentes en memoria; no incluyas la carga en ese tiempo. Usa el mismo orden de columnas, separador, nulos y compresión, y documenta opciones distintas que no puedas igualar.

Usa `time.perf_counter()` inmediatamente antes y después de cada operación. Realiza una ejecución inicial separada y cinco repeticiones, alternando el orden de motores. Escribe cada repetición en una ruta nueva; no acumules archivos dentro de un dataset particionado ya existente. Deja impresión, validación e inventario fuera del intervalo medido.

**Control:** relee los resultados fuera de la medición y verifica equivalencia. Conserva tiempos individuales y mediana. Documenta versiones, CPU, RAM disponible y ubicación de archivos. No describas una lectura como fría sin haber controlado la caché del sistema.

In [7]:
# ---------- Entorno de la medición ----------
import pyarrow as pa
try:
    import psutil
    ram_total_gb = psutil.virtual_memory().total / 1e9
    ram_disp_gb = psutil.virtual_memory().available / 1e9
except ImportError:
    ram_total_gb = ram_disp_gb = None
    try:
        with open("/proc/meminfo") as f:
            info = dict(linea.split(":", 1) for linea in f.read().splitlines())
        ram_total_gb = int(info["MemTotal"].split()[0]) / 1e6
        ram_disp_gb = int(info["MemAvailable"].split()[0]) / 1e6
    except OSError:
        pass
try:
    hilos_polars = pl.thread_pool_size()
except AttributeError:
    hilos_polars = None

ENTORNO = {
    "sistema": platform.platform(), "procesador": platform.processor() or "no reportado",
    "cpu_logicas": os.cpu_count(), "hilos_polars": hilos_polars,
    "ram_total_gb": None if ram_total_gb is None else round(ram_total_gb, 1),
    "ram_disponible_gb": None if ram_disp_gb is None else round(ram_disp_gb, 1),
    "python": platform.python_version(), "pandas": pd.__version__, "polars": pl.__version__,
    "pyarrow": pa.__version__, "geopandas": gpd.__version__,
    "ubicacion_archivos": str(OUT),
}
display(pd.Series(ENTORNO, name="valor").to_frame())

,valor
sistema,Linux-6.18.54.1-microsoft-standard-WSL2-x86_64...
procesador,no reportado
cpu_logicas,24
hilos_polars,24
ram_total_gb,8.1
ram_disponible_gb,4.8
python,3.12.15
pandas,2.2.3
polars,1.35.2
pyarrow,19.0.1


In [8]:
# ---------- Lectura completa: CSV y Parquet con ambos motores ----------
BENCH = Path(OUT) / ("bench_io_" + STAMP)
BENCH.mkdir(parents=True, exist_ok=False)
ruta_csv = EJECUCION / "proyeccion.csv"
ruta_pq = EJECUCION / "proyeccion.parquet"
N_FILAS = len(proyeccion_pd)

mediciones_io, iniciales_io, verificaciones_lectura = [], [], []
for repeticion in range(0, 6):                    # 0 = ejecución inicial (se guarda aparte); 1..5 = repeticiones
    orden = ["pandas", "polars"] if repeticion % 2 == 0 else ["polars", "pandas"]     # alternancia
    for formato in ["csv", "parquet"]:
        for motor in orden:
            if motor == "pandas" and formato == "csv":
                t0 = time.perf_counter()
                d = pd.read_csv(ruta_csv, sep=",", encoding="utf-8", dtype=TIPOS_PD,
                                keep_default_na=False, na_values=[""])
                t1 = time.perf_counter()
            elif motor == "pandas":
                t0 = time.perf_counter()
                d = pd.read_parquet(ruta_pq, engine="pyarrow").astype(TIPOS_PD)
                t1 = time.perf_counter()
            elif formato == "csv":
                t0 = time.perf_counter()
                d = pl.read_csv(ruta_csv, separator=",", null_values=[""], schema_overrides=TIPOS_PL)
                t1 = time.perf_counter()
            else:
                t0 = time.perf_counter()
                d = pl.read_parquet(ruta_pq).with_columns([pl.col(c).cast(t) for c, t in TIPOS_PL.items()])
                t1 = time.perf_counter()
            # --- fuera del intervalo: registro y verificación ---
            filas_salida = len(d) if motor == "pandas" else d.height
            dp = d if motor == "pandas" else a_pandas(d)
            try:
                pd.testing.assert_frame_equal(dp.sort_values("id_fila").reset_index(drop=True), referencia)
                equivalente = True
            except AssertionError:
                equivalente = False
            verificaciones_lectura.append({"motor": motor, "formato": formato, "repeticion": repeticion, "equivalente": equivalente})
            fila = {"motor": motor, "version": pd.__version__ if motor == "pandas" else pl.__version__,
                    "operacion": "lectura_completa", "formato": formato, "particionamiento": "archivo_unico",
                    "repeticion": repeticion, "segundos": t1 - t0, "filas_entrada": N_FILAS, "filas_salida": filas_salida}
            (iniciales_io if repeticion == 0 else mediciones_io).append(fila)

CONTROLES["lectura_equivalente_en_todas_las_corridas"] = all(v["equivalente"] for v in verificaciones_lectura)
print("Lecturas equivalentes al contrato:", CONTROLES["lectura_equivalente_en_todas_las_corridas"])

Lecturas equivalentes al contrato: True


In [9]:
# ---------- Escritura desde datos ya residentes en memoria ----------
bytes_escritos = []
for repeticion in range(0, 6):
    orden = ["polars", "pandas"] if repeticion % 2 == 0 else ["pandas", "polars"]      # empieza distinto a la lectura
    for formato in ["csv", "parquet"]:
        for motor in orden:
            ruta = BENCH / f"{motor}_{formato}_rep{repeticion}.{formato}"              # ruta nueva en cada repetición
            if motor == "pandas" and formato == "csv":
                t0 = time.perf_counter()
                proyeccion_pd.to_csv(ruta, sep=",", na_rep="", index=False, encoding="utf-8")
                t1 = time.perf_counter()
            elif motor == "pandas":
                t0 = time.perf_counter()
                proyeccion_pd.to_parquet(ruta, engine="pyarrow", compression="snappy", index=False)
                t1 = time.perf_counter()
            elif formato == "csv":
                t0 = time.perf_counter()
                proyeccion_pl.write_csv(ruta, separator=",", null_value="")
                t1 = time.perf_counter()
            else:
                t0 = time.perf_counter()
                proyeccion_pl.write_parquet(ruta, compression="snappy")
                t1 = time.perf_counter()
            # --- fuera del intervalo: relectura con el lector común y verificación ---
            if formato == "csv":
                releido = pd.read_csv(ruta, sep=",", dtype=TIPOS_PD, keep_default_na=False, na_values=[""])
            else:
                releido = pd.read_parquet(ruta, engine="pyarrow").astype(TIPOS_PD)
            try:
                pd.testing.assert_frame_equal(releido.sort_values("id_fila").reset_index(drop=True), referencia)
                equivalente = True
            except AssertionError:
                equivalente = False
            bytes_escritos.append({"motor": motor, "formato": formato, "repeticion": repeticion,
                                   "bytes": ruta.stat().st_size, "equivalente": equivalente})
            fila = {"motor": motor, "version": pd.__version__ if motor == "pandas" else pl.__version__,
                    "operacion": "escritura", "formato": formato, "particionamiento": "archivo_unico",
                    "repeticion": repeticion, "segundos": t1 - t0, "filas_entrada": N_FILAS, "filas_salida": len(releido)}
            (iniciales_io if repeticion == 0 else mediciones_io).append(fila)

bytes_escritos = pd.DataFrame(bytes_escritos)
CONTROLES["escritura_equivalente_en_todas_las_corridas"] = bool(bytes_escritos["equivalente"].all())
print("Escrituras equivalentes al contrato:", CONTROLES["escritura_equivalente_en_todas_las_corridas"])

# ---------- Tiempos individuales y mediana (solo repeticiones 1-5) ----------
io = pd.DataFrame(mediciones_io)
display(io.pivot_table(index=["operacion", "formato", "motor"], columns="repeticion", values="segundos"))
resumen_io = (io.groupby(["operacion", "formato", "motor"])["segundos"]
              .agg(mediana="median", minimo="min", maximo="max").reset_index())
display(resumen_io)
display(bytes_escritos.groupby(["formato", "motor"])["bytes"].agg(["min", "max"]))
print("Ejecuciones iniciales (aparte):")
display(pd.DataFrame(iniciales_io)[["motor", "operacion", "formato", "segundos"]])

Escrituras equivalentes al contrato: True


repeticion                              1         2         3         4  \
operacion        formato motor                                            
escritura        csv     pandas  0.208170  0.601729  0.585644  0.605364   
                         polars  0.280963  0.145831  0.149915  0.142734   
                 parquet pandas  0.105836  0.086116  0.096842  0.085340   
                         polars  0.044920  0.061227  0.053166  0.068523   
lectura_completa csv     pandas  0.059416  0.067787  0.060934  0.061401   
                         polars  0.032447  0.061231  0.037510  0.087370   
                 parquet pandas  0.022574  0.022165  0.020368  0.022599   
                         polars  0.022190  0.018616  0.020629  0.019236   

repeticion                              5  
operacion        formato motor             
escritura        csv     pandas  0.727375  
                         polars  0.172282  
                 parquet pandas  0.093701  
                         polars  0.047989  
lectura_completa csv     pandas  0.083597  
                         polars  0.063050  
                 parquet pandas  0.019922  
                         polars  0.027846

,operacion,formato,motor,mediana,minimo,maximo
0,escritura,csv,pandas,0.601729,0.208170,0.727375
1,escritura,csv,polars,0.149915,0.142734,0.280963
2,escritura,parquet,pandas,0.093701,0.085340,0.105836
3,escritura,parquet,polars,0.053166,0.044920,0.068523
4,lectura_completa,csv,pandas,0.061401,0.059416,0.083597
5,lectura_completa,csv,polars,0.061231,0.032447,0.087370
6,lectura_completa,parquet,pandas,0.022165,0.019922,0.022599
7,lectura_completa,parquet,polars,0.020629,0.018616,0.027846


min      max
formato motor                   
csv     pandas  2749328  2749328
        polars  2749328  2749328
parquet pandas   830508   830508
        polars   761659   761659

Ejecuciones iniciales (aparte):


,motor,operacion,formato,segundos
0,pandas,lectura_completa,csv,0.068103
1,polars,lectura_completa,csv,0.039586
2,pandas,lectura_completa,parquet,0.024133
3,polars,lectura_completa,parquet,0.025284
4,polars,escritura,csv,0.059436
5,pandas,escritura,csv,0.200857
6,polars,escritura,parquet,0.023864
7,pandas,escritura,parquet,0.050634


**Interpretación:**
Medí con `time.perf_counter()` justo antes y después de cada llamada directa (`pd.read_csv`/`pl.read_csv`, `pd.read_parquet`/`pl.read_parquet`, `to_csv`/`write_csv`, `to_parquet`/`write_parquet`). Las lecturas incluyen las conversiones necesarias para cumplir el contrato (tipos en `read_csv`, `astype` o `cast` tras leer Parquet); las escrituras parten de datos equivalentes ya cargados y no incluyen la carga. Hubo una ejecución inicial separada y cinco repeticiones, alternando el orden de los motores; cada escritura va a una ruta nueva. La relectura y la comparación con la proyección se hacen fuera del intervalo, y todas las corridas fueron equivalentes en contenido y tipos.

Opciones que no pude igualar: el booleano se escribe `True/False` en pandas y `true/false` en Polars (ambos lectores aceptan las dos formas); el tamaño de grupo de filas, las estadísticas y los metadatos de Parquet son distintos por defecto (pandas añade metadatos propios), y esto cambia los bytes aunque se use Snappy en ambos; la escritura de CSV difiere en detalles de formato de decimales.

**Límites:** los tiempos son de este equipo, estos datos y esta sesión. No llamo "fría" a ninguna lectura porque no controlé la caché del sistema: la segunda y siguientes lecturas probablemente se benefician de ella. Si `REPO` está bajo `/mnt/c` (WSL), la E/S pasa por el sistema de archivos de Windows y puede dominar el tiempo. Con pocas repeticiones, diferencias pequeñas entre motores no son concluyentes; por eso se guardan todos los tiempos y no solo la mediana.

## 4. Procesamiento y consulta desde disco · 40 minutos

Separa dos experimentos:

1. **En memoria:** mide solo el filtro y la agregación sobre tablas ya cargadas en pandas y Polars.
2. **Desde disco:** mide hasta obtener el resultado del filtro y la agregación. Compara archivo único, archivos por año y archivos por año/departamento.

Para las partes, lee el inventario CSV dentro del intervalo medido, filtra sus claves con pandas y obtiene la lista de rutas. Para un año solo, selecciona sus partes y aplica el departamento sobre los registros leídos; para año/departamento, selecciona ambas claves. Usa exactamente la misma lista de archivos para los dos motores. Registra qué rutas seleccionaste. La tabla de inventario es pequeña y se usa igual en ambos recorridos.

En pandas, muestra un bucle que llame a `pd.read_parquet` para cada ruta y luego `pd.concat`. En Polars, muestra un bucle con `pl.read_parquet` y luego `pl.concat`. En cada caso, aplica las mismas condiciones analíticas y agregaciones. El ejemplo siguiente explica la lectura pandas; escribe su equivalente Polars:

```python
partes_pd = []
for ruta in rutas_seleccionadas:
    parte_pd = pd.read_parquet(
        ruta,
        engine="pyarrow",
        columns=["id_fila", "anio", "codigo_departamento",
                 "codigo_municipio", "ph", "ph_valido"]
    )
    partes_pd.append(parte_pd)
seleccion_pd = pd.concat(partes_pd, ignore_index=True)
```

Antes de medir, verifica que la consulta elegida tiene archivos candidatos. Si no los tiene, presenta un resultado vacío con esquema explícito; no llames `concat` sobre una lista vacía.

**Consulta diferida adicional:** sobre el Parquet único usa `pl.scan_parquet`, selección de columnas, filtro, agregación y `collect()`. Mide la ejecución completa y muestra `explain()` fuera del tiempo medido. Contrasta esta variante con `pl.read_parquet`; así separas el efecto de la consulta diferida del de dividir archivos.

**Control:** cinco repeticiones por variante, mismos datos y resultados materializados. Incluye lectura del inventario, selección de rutas, lectura de partes, concatenación, filtro y agregación en el experimento desde disco. Excluye impresión y comprobaciones. Los bytes de archivos seleccionados no son una medida de bytes físicos leídos. No compares construir un plan con ejecutarlo.

In [10]:
# ---------- Experimento 1: en memoria (solo filtro + agregación) ----------
mediciones_proc, iniciales_proc = [], []
for repeticion in range(0, 6):
    orden = ["pandas", "polars"] if repeticion % 2 == 0 else ["polars", "pandas"]
    for motor in orden:
        if motor == "pandas":
            t0 = time.perf_counter()
            f = proyeccion_pd.loc[proyeccion_pd["anio"].eq(ANIO_SEL).fillna(False).astype(bool) & proyeccion_pd["codigo_departamento"].eq(DEP_SEL).fillna(False).astype(bool)]
            f = f.assign(ph_para_media=f["ph"].where(f["ph_valido"]))
            res = (f.groupby("codigo_municipio", dropna=False, sort=True)
                   .agg(total_analisis=("id_fila", "size"), n_ph_validos=("ph_valido", "sum"),
                        ph_media_valido=("ph_para_media", "mean")).reset_index())
            t1 = time.perf_counter()
            res_mem_pd = res
        else:
            t0 = time.perf_counter()
            res = (proyeccion_pl.filter((pl.col("anio") == ANIO_SEL) & (pl.col("codigo_departamento") == DEP_SEL))
                   .group_by("codigo_municipio")
                   .agg(pl.len().alias("total_analisis"), pl.col("ph_valido").sum().alias("n_ph_validos"),
                        pl.col("ph").filter(pl.col("ph_valido")).mean().alias("ph_media_valido"))
                   .sort("codigo_municipio", nulls_last=True))
            t1 = time.perf_counter()
            res_mem_pl = res
        # --- fuera del intervalo ---
        res_cmp = res if motor == "pandas" else a_pandas(res)
        ok = all(comparar_resultados(res_pd, res_cmp).values())
        fila = {"motor": motor, "version": pd.__version__ if motor == "pandas" else pl.__version__,
                "operacion": "filtro_agregacion_memoria", "formato": "en_memoria", "particionamiento": "no_aplica",
                "repeticion": repeticion, "segundos": t1 - t0, "filas_entrada": N_FILAS,
                "filas_salida": len(res) if motor == "pandas" else res.height, "equivalente": ok}
        (iniciales_proc if repeticion == 0 else mediciones_proc).append(fila)

CONTROLES["memoria_resultados_equivalentes"] = all(m["equivalente"] for m in mediciones_proc + iniciales_proc)
print("Experimento en memoria equivalente a la consulta de referencia:", CONTROLES["memoria_resultados_equivalentes"])

NameError: name 'ANIO_SEL' is not defined

In [ ]:
# ---------- Experimento 2: desde disco (lectura + selección + filtro + agregación) ----------
RUTAS = {}                      # rutas seleccionadas por variante y motor (registro)
VARIANTES = [("parquet_unico", ["pandas", "polars", "polars_diferida"]),
             ("por_anio", ["pandas", "polars"]),
             ("por_anio_departamento", ["pandas", "polars"])]

# Verificación previa: ¿hay archivos candidatos para la consulta elegida?
for variante in ["por_anio", "por_anio_departamento"]:
    inv_prev = pd.read_csv(EJECUCION / f"inventario_{variante}.csv",
                           dtype={"anio": "Int64", "codigo_departamento": "string", "archivo": "string"},
                           keep_default_na=False, na_values=[""])
    cond = inv_prev["anio"].eq(ANIO_SEL).fillna(False).astype(bool)
    if variante == "por_anio_departamento":
        cond = cond & inv_prev["codigo_departamento"].eq(DEP_SEL).fillna(False).astype(bool)
    print(variante, "-> archivos candidatos:", int(cond.sum()))

for repeticion in range(0, 6):
    for variante, motores in VARIANTES:
        giro = repeticion % len(motores)
        for motor in motores[giro:] + motores[:giro]:                 # alternancia del orden
            t0 = time.perf_counter()
            if variante != "parquet_unico":
                # el inventario CSV es pequeño y se lee igual para ambos motores (con pandas)
                inv = pd.read_csv(EJECUCION / f"inventario_{variante}.csv",
                                  dtype={"anio": "Int64", "codigo_departamento": "string", "archivo": "string"},
                                  keep_default_na=False, na_values=[""])
                sel = inv["anio"].eq(ANIO_SEL).fillna(False).astype(bool)
                if variante == "por_anio_departamento":
                    sel = sel & inv["codigo_departamento"].eq(DEP_SEL).fillna(False).astype(bool)
                rutas = [EJECUCION / r for r in inv.loc[sel, "archivo"]]

            if motor == "pandas":
                if variante == "parquet_unico":
                    t = pd.read_parquet(ruta_pq, engine="pyarrow", columns=COLS)
                elif len(rutas) > 0:
                    partes_pd = []
                    for ruta in rutas:
                        parte_pd = pd.read_parquet(ruta, engine="pyarrow", columns=COLS)
                        partes_pd.append(parte_pd)
                    t = pd.concat(partes_pd, ignore_index=True)
                else:                                                  # sin candidatos: tabla vacía con esquema explícito
                    t = pd.DataFrame({c: pd.Series(dtype=TIPOS_PD[c]) for c in COLS})
                f = t.loc[t["anio"].eq(ANIO_SEL).fillna(False).astype(bool) & t["codigo_departamento"].eq(DEP_SEL).fillna(False).astype(bool)]
                f = f.assign(ph_para_media=f["ph"].where(f["ph_valido"]))
                res = (f.groupby("codigo_municipio", dropna=False, sort=True)
                       .agg(total_analisis=("id_fila", "size"), n_ph_validos=("ph_valido", "sum"),
                            ph_media_valido=("ph_para_media", "mean")).reset_index())
                n_leidas = len(t)
            elif motor == "polars":
                if variante == "parquet_unico":
                    t = pl.read_parquet(ruta_pq, columns=COLS)
                elif len(rutas) > 0:
                    partes_pl = []
                    for ruta in rutas:
                        parte_pl = pl.read_parquet(ruta, columns=COLS)
                        partes_pl.append(parte_pl)
                    t = pl.concat(partes_pl)
                else:
                    t = pl.DataFrame(schema={c: TIPOS_PL[c] for c in COLS})
                res = (t.filter((pl.col("anio") == ANIO_SEL) & (pl.col("codigo_departamento") == DEP_SEL))
                       .group_by("codigo_municipio")
                       .agg(pl.len().alias("total_analisis"), pl.col("ph_valido").sum().alias("n_ph_validos"),
                            pl.col("ph").filter(pl.col("ph_valido")).mean().alias("ph_media_valido"))
                       .sort("codigo_municipio", nulls_last=True))
                n_leidas = t.height
            else:                                                      # polars_diferida: scan_parquet + collect()
                res = (pl.scan_parquet(ruta_pq).select(COLS)
                       .filter((pl.col("anio") == ANIO_SEL) & (pl.col("codigo_departamento") == DEP_SEL))
                       .group_by("codigo_municipio")
                       .agg(pl.len().alias("total_analisis"), pl.col("ph_valido").sum().alias("n_ph_validos"),
                            pl.col("ph").filter(pl.col("ph_valido")).mean().alias("ph_media_valido"))
                       .sort("codigo_municipio", nulls_last=True)
                       .collect())
                n_leidas = N_FILAS          # filas del archivo; el plan puede leer menos
            t1 = time.perf_counter()

            # --- fuera del intervalo: registro y verificación ---
            if variante != "parquet_unico":
                RUTAS[(variante, motor)] = [str(r.relative_to(EJECUCION)) for r in rutas]
            es_pd = motor == "pandas"
            ok = all(comparar_resultados(res_pd, res if es_pd else a_pandas(res)).values())
            fila = {"motor": "pandas" if es_pd else "polars",
                    "version": pd.__version__ if es_pd else pl.__version__,
                    "operacion": "filtro_agregacion_disco_diferida" if motor == "polars_diferida" else "filtro_agregacion_disco",
                    "formato": "parquet", "particionamiento": variante, "repeticion": repeticion,
                    "segundos": t1 - t0, "filas_entrada": n_leidas,
                    "filas_salida": len(res) if es_pd else res.height, "equivalente": ok}
            (iniciales_proc if repeticion == 0 else mediciones_proc).append(fila)

CONTROLES["disco_resultados_equivalentes"] = all(m["equivalente"] for m in mediciones_proc + iniciales_proc)
for variante in ["por_anio", "por_anio_departamento"]:
    CONTROLES[f"{variante}_misma_lista_de_archivos_en_ambos_motores"] = RUTAS[(variante, "pandas")] == RUTAS[(variante, "polars")]
    print(variante, "-> rutas seleccionadas:", RUTAS[(variante, "pandas")])
print("Experimento desde disco equivalente a la referencia:", CONTROLES["disco_resultados_equivalentes"])

In [ ]:
# ---------- Plan de la consulta diferida (fuera del tiempo medido) ----------
consulta_diferida = (pl.scan_parquet(ruta_pq).select(COLS)
                     .filter((pl.col("anio") == ANIO_SEL) & (pl.col("codigo_departamento") == DEP_SEL))
                     .group_by("codigo_municipio")
                     .agg(pl.len().alias("total_analisis"), pl.col("ph_valido").sum().alias("n_ph_validos"),
                          pl.col("ph").filter(pl.col("ph_valido")).mean().alias("ph_media_valido"))
                     .sort("codigo_municipio", nulls_last=True))
print(consulta_diferida.explain())

proc = pd.DataFrame(mediciones_proc)
resumen_proc = (proc.groupby(["operacion", "particionamiento", "motor"])["segundos"]
                .agg(mediana="median", minimo="min", maximo="max").reset_index())
display(proc.pivot_table(index=["operacion", "particionamiento", "motor"], columns="repeticion", values="segundos"))
display(resumen_proc)
print("Ejecuciones iniciales (aparte):")
display(pd.DataFrame(iniciales_proc)[["motor", "operacion", "particionamiento", "segundos"]])

**Interpretación:**
Separé dos experimentos. **En memoria** mido únicamente el filtro y la agregación sobre las tablas ya cargadas. **Desde disco** mido todo el camino hasta tener el resultado materializado: leer el inventario CSV, seleccionar rutas con pandas, leer cada parte con `pd.read_parquet` / `pl.read_parquet` en un bucle, concatenar, filtrar y agregar. Ambos motores usan exactamente la misma lista de archivos (el control lo comprueba y las rutas quedan registradas); para el diseño por año se filtra el departamento sobre los registros leídos, y para año/departamento se seleccionan las dos claves. Si no hubiera archivos candidatos, el código genera una tabla vacía con esquema explícito en lugar de llamar `concat` con una lista vacía.

La **consulta diferida** (`pl.scan_parquet` + `select` + `filter` + `group_by` + `collect()`) se mide ejecutando el plan completo; el `explain()` se muestra aparte, así no se compara construir un plan con ejecutarlo. Contrastarla con `pl.read_parquet` sobre el mismo archivo único separa el efecto de la evaluación diferida del efecto de dividir archivos. Todas las variantes se verificaron contra la consulta de referencia de la sección 1.

**Límites:** los bytes de los archivos seleccionados no son bytes físicos leídos (Parquet puede saltar columnas o grupos de filas). Las cinco repeticiones comparten caché del sistema, así que la lectura no es "fría". Con pocos datos, la diferencia entre variantes puede ser de milisegundos y quedar dentro del ruido; esos casos se describen como diferencias pequeñas o variables, no como victorias.

## 5. Llevar los resultados tabulares a un mapa · 30 minutos

Lee directamente la capa `municipios` del GeoPackage de PD04. Muestra columnas, CRS y cantidad de geometrías. Selecciona únicamente código y geometría para evitar reutilizar por accidente los indicadores del año anterior.

Convierte el resultado agregado de Polars con `.to_pandas()` después de medir. Une por separado el resultado pandas y el resultado Polars con los mismos polígonos mediante `merge(..., how="left", validate="one_to_one")`. Conserva el código como texto y verifica antes la unicidad. Presenta resultados sin geometría y polígonos sin resultado.

Genera dos mapas del departamento y año elegidos, uno por motor, usando igual extensión, columna, límites de color, leyenda y tratamiento de ausentes. Guarda `comparacion_mapas.png`. Comprueba numéricamente las claves, conteos y medias antes de juzgar la igualdad visual.

**Control:** las geometrías no participan en el benchmark tabular ni se convierten a texto para medirlas con Polars. La unión y la representación son trabajo de GeoPandas fuera de esos tiempos. Un mapa igual no sustituye una comparación de valores. La cantidad de registros no demuestra cobertura completa de suelos.

In [ ]:
geometrias = gpd.read_file(
    REPO + "/kit/salidas/soluciones/PD04/resultados_espaciales.gpkg",
    layer="municipios",
    engine="pyogrio"
)
display(geometrias.head())
print(geometrias.shape, geometrias.crs)

In [ ]:
from matplotlib.patches import Patch

# ---------- 5.1 Resultados de los dos motores como pandas (después de medir) ----------
geom = geometrias[["codigo_municipio", "geometry"]].copy()          # solo código y geometría
tipo_codigo = geom["codigo_municipio"].dtype
resultado_pd_mapa = res_mem_pd.copy()
resultado_pl_mapa = res_mem_pl.to_pandas()                          # conversión de Polars a pandas, fuera de los tiempos
for r in (resultado_pd_mapa, resultado_pl_mapa):
    r["codigo_municipio"] = r["codigo_municipio"].astype(tipo_codigo)

print("Códigos únicos -> geometrías:", geom["codigo_municipio"].is_unique,
      "| pandas:", resultado_pd_mapa["codigo_municipio"].is_unique,
      "| Polars:", resultado_pl_mapa["codigo_municipio"].is_unique)

# ---------- 5.2 Uniones left con las mismas geometrías ----------
mapa_pd = geom.merge(resultado_pd_mapa, on="codigo_municipio", how="left", validate="one_to_one", indicator=True)
mapa_pl = geom.merge(resultado_pl_mapa, on="codigo_municipio", how="left", validate="one_to_one", indicator=True)
for nombre, mapa, res_m in [("pandas", mapa_pd, resultado_pd_mapa), ("Polars", mapa_pl, resultado_pl_mapa)]:
    sin_geom = res_m[~res_m["codigo_municipio"].isin(geom["codigo_municipio"])]
    print(f"[{nombre}] polígonos: {len(mapa)} | polígonos sin resultado: {int((mapa['_merge'] == 'left_only').sum())} "
          f"| resultados sin geometría: {len(sin_geom)} (análisis: {int(sin_geom['total_analisis'].sum())})")
    if len(sin_geom):
        display(sin_geom)

# ---------- 5.3 Comparación numérica ANTES del juicio visual ----------
vals = ["total_analisis", "n_ph_validos", "ph_media_valido"]
a = mapa_pd.sort_values("codigo_municipio").reset_index(drop=True)
b = mapa_pl.sort_values("codigo_municipio").reset_index(drop=True)
CONTROLES["mapa_mismas_claves"] = bool(a["codigo_municipio"].equals(b["codigo_municipio"]))
CONTROLES["mapa_conteos_exactos"] = bool(np.array_equal(a[["total_analisis", "n_ph_validos"]].to_numpy(dtype=float),
                                                        b[["total_analisis", "n_ph_validos"]].to_numpy(dtype=float), equal_nan=True))
CONTROLES["mapa_medias_atol_1e-6"] = bool(np.allclose(a["ph_media_valido"].astype(float), b["ph_media_valido"].astype(float),
                                                      rtol=0, atol=1e-6, equal_nan=True))
print({k: v for k, v in CONTROLES.items() if k.startswith("mapa_")})

# ---------- 5.4 Dos mapas con la misma extensión, escala y tratamiento de ausentes ----------
dep_pd = a[a["codigo_municipio"].str[:2] == DEP_SEL]
dep_pl = b[b["codigo_municipio"].str[:2] == DEP_SEL]
vmin = float(min(dep_pd["total_analisis"].min(), dep_pl["total_analisis"].min()))
vmax = float(max(dep_pd["total_analisis"].max(), dep_pl["total_analisis"].max()))
xmin, ymin, xmax, ymax = dep_pd.total_bounds

fig, ejes = plt.subplots(1, 2, figsize=(14, 7))
for ax, mapa_dep, titulo in zip(ejes, [dep_pd, dep_pl], ["pandas", "Polars"]):
    mapa_dep.plot(column="total_analisis", cmap="YlGnBu", vmin=vmin, vmax=vmax, edgecolor="white", linewidth=0.4,
                  legend=True, legend_kwds={"label": "Total de análisis", "shrink": 0.6},
                  missing_kwds={"color": "lightgrey", "edgecolor": "white"}, ax=ax)
    ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
    ax.set_title(f"{titulo} - departamento {DEP_SEL}, {ANIO_SEL}")
    ax.set_axis_off()
    ax.legend(handles=[Patch(facecolor="lightgrey", edgecolor="white", label="Sin resultado (no implica cero)")],
              loc="upper center", bbox_to_anchor=(0.5, 0.0))
plt.savefig(OUT + "/comparacion_mapas.png", dpi=150, bbox_inches="tight")
plt.show()

**Interpretación:**
Leí la capa `municipios` del GeoPackage de PD04 y conservé solo `codigo_municipio` y la geometría, para no arrastrar indicadores del año anterior. El resultado agregado de Polars se convirtió con `.to_pandas()` después de medir, y cada resultado se unió por separado con las mismas geometrías mediante `merge(..., how="left", validate="one_to_one")`, con el código como texto y su unicidad verificada de antemano. Registré los resultados sin geometría (por ejemplo, el grupo de código ausente) y los polígonos sin resultado.

Antes de mirar los mapas comparé numéricamente claves, conteos exactos y medias (tolerancia absoluta 1e-6, relativa 0). Los dos mapas usan la misma extensión, columna, límites de color (`vmin`/`vmax` compartidos), leyenda y gris para los ausentes.

**Límites:** un mapa igual no sustituye la comparación de valores; dos mapas pueden verse idénticos con diferencias menores que la escala de color. Un polígono gris significa "sin resultado en esta consulta", no "cero análisis" en el territorio. El número de registros no demuestra cobertura completa de los suelos. Las geometrías no participaron en ningún tiempo del benchmark tabular: la unión y el dibujo son trabajo de GeoPandas fuera de las mediciones.

## 6. Reejecución, evidencia y decisión · 45 minutos

Repite la construcción de los archivos en otra carpeta de ejecución nueva. No borres carpetas de datos originales ni reutilices un directorio particionado con archivos viejos. Comprueba contenido y total de registros; el nombre de los archivos o su hash no tiene por qué coincidir para que el contenido sea equivalente.

Guarda `mediciones.csv` con motor, versión, operación, formato, particionamiento, repetición, segundos, filas de entrada y filas de salida. Guarda aparte la ejecución inicial; calcula la mediana solo con las cinco repeticiones medidas. Añade inventario de bytes/archivos y controles de equivalencia.

Responde: ¿qué parte mejora con Parquet?, ¿cuándo ayuda la selección de particiones?, ¿qué diseño genera archivos demasiado pequeños?, ¿qué cambia entre tiempo en memoria y tiempo desde disco?, ¿qué elegirías para este conjunto y por qué?

**Control:** no presupongas que Polars o el particionamiento ganarán. Diferencias pequeñas o variables deben describirse como tales. El tamaño de un DataFrame no equivale a la memoria máxima del proceso.

In [ ]:
# ---------- Construcción de las cuatro representaciones ----------
EJECUCION_B = Path(OUT) / ("ejecucion_" + STAMP + "_b")
EJECUCION_B.mkdir(parents=True, exist_ok=False)            # carpeta nueva en cada construcción

# (a) CSV único: coma, UTF-8, nulos como vacío
proyeccion_pd.to_csv(EJECUCION_B / "proyeccion.csv", sep=",", na_rep="", index=False, encoding="utf-8")
# (b) Parquet único
proyeccion_pd.to_parquet(EJECUCION_B / "proyeccion.parquet", engine="pyarrow", compression="snappy", index=False)

# (c) por año y (d) por año y departamento: un groupby divide, cada grupo es un archivo
DISENOS = {"por_anio": ["anio"], "por_anio_departamento": ["anio", "codigo_departamento"]}
for nombre_diseno, claves in DISENOS.items():
    carpeta = EJECUCION_B / nombre_diseno
    carpeta.mkdir(parents=True, exist_ok=False)
    inventario = []
    for numero, (valores, grupo) in enumerate(proyeccion_pd.groupby(claves, dropna=False, sort=True)):
        valores = valores if isinstance(valores, tuple) else (valores,)
        archivo = carpeta / f"parte-{numero:04d}.parquet"
        grupo.to_parquet(archivo, engine="pyarrow", compression="snappy", index=False)
        fila = dict(zip(claves, valores))                                  # claves originales (pueden ser ausentes)
        fila.update({"archivo": str(archivo.relative_to(EJECUCION_B)),        # ruta relativa
                     "filas": len(grupo), "bytes": archivo.stat().st_size})
        inventario.append(fila)
    inv = pd.DataFrame(inventario)
    inv["anio"] = inv["anio"].astype("Int64")
    if "codigo_departamento" in inv.columns:
        inv["codigo_departamento"] = inv["codigo_departamento"].astype("string")
    inv.to_csv(EJECUCION_B / f"inventario_{nombre_diseno}.csv", index=False)
    print(nombre_diseno, "->", len(inv), "archivos,", int(inv["filas"].sum()), "filas")

In [ ]:
# ---------- Comparación de la segunda ejecución con la primera ----------
for nombre in DISENOS:
    inv_a, reunido_a = leer_diseno(EJECUCION, nombre)
    inv_b, reunido_b = leer_diseno(EJECUCION_B, nombre)
    pd.testing.assert_frame_equal(reunido_a.sort_values("id_fila").reset_index(drop=True),
                                  reunido_b.sort_values("id_fila").reset_index(drop=True))
    CONTROLES[f"reejecucion_{nombre}_contenido_igual"] = True
    CONTROLES[f"reejecucion_{nombre}_mismas_filas"] = len(reunido_a) == len(reunido_b) == len(proyeccion_pd)
    CONTROLES[f"reejecucion_{nombre}_mismos_ids"] = reunido_a["id_fila"].is_unique and reunido_b["id_fila"].is_unique \
        and set(reunido_a["id_fila"]) == set(reunido_b["id_fila"])
    CONTROLES[f"reejecucion_{nombre}_mismo_numero_archivos"] = len(inv_a) == len(inv_b)
pd.testing.assert_frame_equal(pd.read_parquet(EJECUCION / "proyeccion.parquet").sort_values("id_fila").reset_index(drop=True),
                              pd.read_parquet(EJECUCION_B / "proyeccion.parquet").sort_values("id_fila").reset_index(drop=True))
CONTROLES["reejecucion_parquet_unico_contenido_igual"] = True
print({k: v for k, v in CONTROLES.items() if k.startswith("reejecucion_")})
print("Carpetas de ejecución distintas:", EJECUCION != EJECUCION_B, "| datos originales intactos:", Path(INPUT).exists())

In [ ]:
# ---------- Exportaciones ----------
COLUMNAS_MED = ["motor", "version", "operacion", "formato", "particionamiento", "repeticion",
                "segundos", "filas_entrada", "filas_salida"]
todas = pd.DataFrame(mediciones_io + mediciones_proc)
iniciales = pd.DataFrame(iniciales_io + iniciales_proc)
todas[COLUMNAS_MED].to_csv(OUT + "/mediciones.csv", index=False)
iniciales[COLUMNAS_MED].to_csv(OUT + "/mediciones_iniciales.csv", index=False)

# Medianas: SOLO con las cinco repeticiones medidas
medianas = (todas.groupby(["operacion", "formato", "particionamiento", "motor"])["segundos"]
            .agg(mediana="median", minimo="min", maximo="max", repeticiones="size").reset_index())
medianas.to_csv(OUT + "/medianas.csv", index=False)

# Inventario de bytes/archivos de ambas construcciones
filas_inv = []
for etiqueta, base in [("ejecucion_a", EJECUCION), ("ejecucion_b", EJECUCION_B)]:
    for nombre in DISENOS:
        inv_x, _ = leer_diseno(base, nombre)
        filas_inv.append({"ejecucion": etiqueta, "diseno": nombre, "n_archivos": len(inv_x),
                          "bytes_totales": int(inv_x["bytes"].sum()), "bytes_min": int(inv_x["bytes"].min()),
                          "bytes_max": int(inv_x["bytes"].max()), "filas_min": int(inv_x["filas"].min())})
    filas_inv.append({"ejecucion": etiqueta, "diseno": "parquet_unico", "n_archivos": 1,
                      "bytes_totales": (base / "proyeccion.parquet").stat().st_size})
    filas_inv.append({"ejecucion": etiqueta, "diseno": "csv_unico", "n_archivos": 1,
                      "bytes_totales": (base / "proyeccion.csv").stat().st_size})
inventario_bytes = pd.DataFrame(filas_inv)
inventario_bytes.to_csv(OUT + "/inventario_bytes.csv", index=False)
bytes_escritos.to_csv(OUT + "/bytes_escritura_benchmark.csv", index=False)

controles_df = pd.DataFrame({"control": list(CONTROLES.keys()), "cumple": [bool(v) for v in CONTROLES.values()]})
controles_df.to_csv(OUT + "/controles_equivalencia.csv", index=False)
pd.Series(ENTORNO, name="valor").to_frame().to_csv(OUT + "/entorno.csv")
print("Controles que NO se cumplen:", controles_df.loc[~controles_df["cumple"], "control"].tolist())
display(medianas)
display(inventario_bytes)
print("Tamaño en memoria (no es el pico del proceso): pandas",
      int(proyeccion_pd.memory_usage(deep=True).sum()), "bytes | Polars", int(proyeccion_pl.estimated_size()), "bytes")

In [ ]:
# ---------- Borrador de respuestas con las cifras de ESTA ejecución ----------
def m(operacion, motor, formato=None, part=None):
    d = todas[(todas["operacion"] == operacion) & (todas["motor"] == motor)]
    if formato is not None:
        d = d[d["formato"] == formato]
    if part is not None:
        d = d[d["particionamiento"] == part]
    return d["segundos"]

def contraste(a, b, nombre_a, nombre_b):
    ma, mb = a.median(), b.median()
    solapan = (a.max() >= b.min()) and (b.max() >= a.min())
    juicio = "rangos solapados: diferencia no distinguible del ruido con 5 repeticiones" if solapan \
        else "rangos separados en esta sesión"
    return f"{nombre_a} {ma*1000:.2f} ms vs {nombre_b} {mb*1000:.2f} ms (razón {ma/mb:.2f}) -> {juicio}"

print("1) ¿Qué mejora con Parquet? (lectura completa, CSV vs Parquet)")
for motor in ["pandas", "polars"]:
    print("  ", motor, ":", contraste(m("lectura_completa", motor, "csv"), m("lectura_completa", motor, "parquet"), "CSV", "Parquet"))
print("   bytes CSV vs Parquet (primera ejecución):",
      inventario_bytes.query("ejecucion == 'ejecucion_a' and diseno in ['csv_unico','parquet_unico']")[["diseno", "bytes_totales"]].to_dict("records"))

print("\n2) ¿Cuándo ayuda la selección de particiones? (consulta desde disco)")
for motor in ["pandas", "polars"]:
    unico = m("filtro_agregacion_disco", motor, "parquet", "parquet_unico")
    for dis in ["por_anio", "por_anio_departamento"]:
        print("  ", motor, dis, ":", contraste(m("filtro_agregacion_disco", motor, "parquet", dis), unico, dis, "parquet_unico"))
print("   consulta diferida vs lectura eager (archivo único):",
      contraste(m("filtro_agregacion_disco_diferida", "polars"), m("filtro_agregacion_disco", "polars", "parquet", "parquet_unico"), "scan+collect", "read_parquet"))

print("\n3) ¿Qué diseño genera archivos demasiado pequeños? (umbral:", UMBRAL_PEQUENO_BYTES, "bytes /", UMBRAL_PEQUENO_FILAS, "filas)")
display(resumen_disenos[resumen_disenos["diseno"].isin(DISENOS.keys())][
    ["diseno", "n_archivos", "bytes_medio", "bytes_min", "filas_min", "archivos_bajo_umbral_bytes", "archivos_bajo_umbral_filas"]])

print("\n4) Memoria vs disco (mediana, filtro + agregación)")
for motor in ["pandas", "polars"]:
    print("  ", motor, ":", contraste(m("filtro_agregacion_memoria", motor), m("filtro_agregacion_disco", motor, "parquet", "parquet_unico"),
                                      "en memoria", "desde disco (único)"))

print("\n5) Candidato con menor mediana desde disco:")
cand = medianas[medianas["operacion"].str.startswith("filtro_agregacion_disco")].sort_values("mediana").head(3)
display(cand)

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## Entrega y referencias de API

`soluciones/PD05_particionamiento_pandas_polars.ipynb`, mediciones, inventarios, comparación cartográfica y controles regenerables en `kit/salidas/soluciones/PD05/`. Incluye una recomendación basada en resultados; no se califica obtener el menor tiempo.

- [pandas: escritura de archivos Parquet](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html)
- [Polars: fuentes y ejecución diferida](https://docs.pola.rs/user-guide/lazy/sources_sinks/)
- [Polars: scan_parquet](https://docs.pola.rs/api/python/stable/reference/api/polars.scan_parquet.html)

Usa las versiones fijadas por el curso; si la documentación más reciente muestra una opción diferente, comprueba su disponibilidad antes de usarla.